In [62]:
import numpy as np
import scipy.stats as stats

In [63]:
n = 100
beta = 0.95
theta = 10
Xn = stats.uniform(loc = theta, scale = theta).rvs(size = n)
print(Xn)

[10.52715725 13.50509269 10.61847591 11.33560177 11.26267986 11.70535426
 15.07481129 14.75923052 15.72260555 16.0207085  11.06424485 12.13690142
 12.02997503 18.92663204 14.59128569 12.38288468 11.75766256 13.73461221
 17.42356817 16.84104968 15.0039211  16.8292217  17.35238132 15.40037137
 11.36866299 16.68392247 16.18481276 18.44165199 16.60812135 18.71840974
 16.10829608 11.95205783 10.38184368 17.05717339 19.40497347 17.24624478
 12.50919383 17.27317054 19.96234141 12.78057922 17.80610305 13.14424986
 13.23698536 10.20276137 18.27073537 14.97684947 16.16670866 11.6914971
 17.39694268 13.53349272 15.00661187 18.37746361 18.73766278 16.44628769
 14.5064094  11.14558719 11.07245686 19.95781552 17.45256643 10.60555575
 17.68289202 14.07727571 10.78583299 15.16450529 15.18248119 16.35891884
 15.83805494 11.76340559 15.2073444  10.08353551 19.73579516 15.07409954
 17.6728736  15.33543518 18.73259445 14.07350675 11.0048702  17.78835448
 15.92332028 17.09771528 17.14324491 19.70766837 12.

In [64]:
x_mid = np.mean(Xn)
x_max = np.max(Xn)
theta1 = 2 / 3 * x_mid
theta2 = (n + 1) / (2 * n + 1) * x_max
mu2 = np.sum([(i - x_mid) ** 2 for i in Xn]) / n

Точный доверительный интервал

In [65]:
l_acc = x_max / (((1 + beta) / 2) ** (1 / n) + 1)
r_acc = x_max / (((1 - beta) / 2) ** (1 / n) + 1)
len_acc = r_acc - l_acc
print("Точный доверительный интервал для theta - ", l_acc, " < theta < ", r_acc)
print("Длина доверительного интервала - ", len_acc)

Точный доверительный интервал для theta -  9.98243420997631  < theta <  10.165246507376505
Длина доверительного интервала -  0.18281229740019533


Асимптотический доверительный интервал

In [66]:
t1 = stats.norm.ppf((1 - beta) / 2)
t2 = stats.norm.ppf((1 + beta) / 2)
l_as = theta1 - 2 / 3 * np.sqrt(mu2 / n) * t2
r_as = theta1 - 2 / 3 * np.sqrt(mu2 / n) * t1
len_as = r_as - l_as
print("Асимптотический доверительный интервал для theta - ", l_as, "< theta <", r_as)
print("Длина доверительного интервала", len_as)


Асимптотический доверительный интервал для theta -  9.579904651678111 < theta < 10.321278415985146
Длина доверительного интервала 0.7413737643070348


Численное построение бутстраповского доверительного интервала

In [67]:
N = 10**3

def btstrp_OMM(x, N):
    arr = []
    n = len(x)
    for _ in range(N):
        sample = np.random.choice(x, size=n, replace=True)
        arr += [2/3*np.mean(sample) - theta1]
    return sorted(np.array(arr))

def btstrp_OMP(x, N):
    arr = []
    n = len(x)
    for _ in range(N):
        sample = np.random.choice(x, size=n, replace=True)
        arr += [(n+1)*np.max(sample)/(2*n+1) - theta2]
    return sorted(np.array(arr))

bs_OMM_arr = btstrp_OMM(Xn, N)
bs_OMP_arr = btstrp_OMP(Xn, N)

t1 = int((1 - beta)/2 * N - 1)
t2 = int((1 + beta)/2 * N - 1)

bs_omm_left = theta1 - bs_OMM_arr[t2]
bs_omm_right = theta1 - bs_OMM_arr[t1]

bs_omp_left = theta2 - bs_OMP_arr[t2]
bs_omp_right = theta2 - bs_OMP_arr[t1]

bs_omm_l = bs_omm_right - bs_omm_left
bs_omp_l = bs_omp_right - bs_omp_left

print("Непараметрический бутстраповский доверительный интервал для theta(ОММ) - ", bs_omm_left, "< theta <", bs_omm_right)
print("Длина доверительного интервала(OMM) - ", bs_omm_l)

print("Непараметрический бутстраповский доверительный интервал для theta(ОМP):", bs_omp_left, "< theta <", bs_omp_right)
print("Длина доверительного интервала(OMP) - ", bs_omp_l)

Непараметрический бутстраповский доверительный интервал для theta(ОММ) -  9.577557625111218 < theta < 10.333456650390987
Длина доверительного интервала(OMM) -  0.7558990252797688
Непараметрический бутстраповский доверительный интервал для theta(ОМP): 10.030828268853847 < theta < 10.144664940454758
Длина доверительного интервала(OMP) -  0.11383667160091093


Сравниваем доверительные интервалы

In [68]:
sort_lens = sorted([(len_acc, "Точный"), (len_as, "Асимптотический"), (bs_omm_l, "Бутстрап ОММ"), (bs_omp_l, "Бутстрап ОМП")])
print("Рейтинг доверительных интервалов:")
for i in range(len(sort_lens)):
    print(f"{i+1})", sort_lens[i][1], f"(l = {np.round(sort_lens[i][0],3)})")

Рейтинг доверительных интервалов:
1) Бутстрап ОМП (l = 0.114)
2) Точный (l = 0.183)
3) Асимптотический (l = 0.741)
4) Бутстрап ОММ (l = 0.756)


In [8]:
for i in range(50):
    print((3 ** i) % 193, i)

1 0
3 1
9 2
27 3
81 4
50 5
150 6
64 7
192 8
190 9
184 10
166 11
112 12
143 13
43 14
129 15
1 16
3 17
9 18
27 19
81 20
50 21
150 22
64 23
192 24
190 25
184 26
166 27
112 28
143 29
43 30
129 31
1 32
3 33
9 34
27 35
81 36
50 37
150 38
64 39
192 40
190 41
184 42
166 43
112 44
143 45
43 46
129 47
1 48
3 49
